# 👗 DressApp Eyes (Gemma-4 E4B) Multimodal Fine-Tuning & Quantization Pipeline

This notebook provides the complete end-to-end pipeline to fine-tune **DressApp Eyes** (`google/gemma-4-e4b-it`) across all fashion workflows using a **Tri-Dataset Hybrid Blend** from Hugging Face:
1. **`ashraq/fashion-product-images-small`**: Clean isolated studio product cutouts for exact taxonomy grounding (Tops, Bottoms, Footwear, Accessories, Outerwear/Dresses).
2. **`Marqo/deepfashion-multimodal`**: High-resolution garments with detailed descriptions for complex patterns (camo, plaid, floral), rich fabric materials (silk, linen, denim, wool), and fluent captions.
3. **`ArtmeScienceLab/Garments2Look`**: 80,000 multi-garment to complete look pairs for full-body outfit coordination (mandatory top+bottom OR dress + footwear) and anatomical ordering.

### Workflow Steps:
1. **Environment Setup & llama.cpp Build**: Install PyTorch, Transformers, PEFT, BitsAndBytes, and compile `llama.cpp` tools (`convert_hf_to_gguf.py` & `llama-quantize`).
2. **Import Assets from VPS**: Connect to the Hetzner VPS (`178.105.144.142`) via SCP to download `mmproj-BF16.gguf` and baseline files.
3. **Tri-Dataset Ingestion & ETL**: Stream and format all three datasets directly into DressApp's canonical JSON schemas (`<garmentVision>`, `<Scheduled Outfit>`, `<Stylist Chat>`).
4. **QLoRA Fine-Tuning**: 4-bit quantized SFT training targeting attention and MLP projections while preserving native vision embeddings.
5. **Quantize to GGUF**: Merge PEFT LoRA into base weights, export intermediate F16 GGUF, and quantize to `Q3_K_M` (~2.85 GB) and `Q4_K_M`.
6. **Evaluation & Smoke Tests**: Test attribute extraction, unique 2-4 word titles, 100% color/material sums, 3-tier gender hierarchy, and outfit coordination.
7. **Push to VPS & Verify**: Securely upload `gemma-4-E4B-it-Q3_K_M.gguf` to `/srv/AI-Stylist/eyes_gguf/` and restart `dressapp-eyes`.


## 🛠️ Step 1: Environment Setup & Tooling
Install deep learning dependencies, Hugging Face streaming libraries, and build `llama.cpp` conversion and quantization tools.


In [ ]:
# Check GPU availability
!nvidia-smi

# Remove incompatible legacy torchao and unused tensorflow to eliminate protobuf collisions
!pip uninstall -y torchao tensorflow

# Force disable TensorFlow in Transformers environment
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

# Install PyTorch with CUDA 12.1 and modern Python 3.13 scientific libraries
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U 'numpy>=2.2.1' 'scipy>=1.15.0' 'pandas>=2.2.3' pyarrow
# Install transformers from git source for native Gemma 4 (Gemma4Processor) architecture support
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U datasets peft 'bitsandbytes>=0.46.1' accelerate pillow paramiko scp gguf sentencepiece protobuf

# Clone and build llama.cpp for GGUF conversion & quantization utilities
import os, sys
if not os.path.exists('llama.cpp'):
    !git clone https://github.com/ggerganov/llama.cpp.git
    !cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=OFF
    !cmake --build llama.cpp/build --config Release -j$(nproc) --target llama-quantize
    !pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

print('✅ Dependencies installed and llama.cpp tools compiled!')

# Automatic runtime restart if NumPy was upgraded in an active session
import numpy as np
np_ver = tuple(map(int, np.__version__.split('.')[:2]))
if np_ver < (2, 2):
    print(f'⚠️ Active NumPy is {np.__version__} (< 2.2). Triggering automatic kernel reload...')
    import os
    os.kill(os.getpid(), 9)
else:
    print(f'✅ NumPy {np.__version__} and Python 3.13 runtime ready!')


## 📥 Step 2: (Optional) Import Assets from Hetzner VPS Server
> **Note:** This step is **OPTIONAL** and can be safely skipped! Proceed directly to Step 3.
> 
> **Why root password fails:** The Hetzner VPS has `PermitRootLogin prohibit-password` enabled for security. Root login is restricted to SSH key pairs only.
> 
> You do **NOT** need `mmproj-BF16.gguf` during training or GGUF quantization. `mmproj-BF16.gguf` already exists on the VPS at `/srv/AI-Stylist/eyes_gguf/mmproj-BF16.gguf` and remains unchanged.


In [ ]:
import os
from pathlib import Path

LOCAL_ASSETS_DIR = Path('vps_assets')
LOCAL_ASSETS_DIR.mkdir(parents=True, exist_ok=True)

# You can safely skip this step! Training does not require VPS connection.
print('ℹ️ Step 2 is OPTIONAL. You can safely proceed to Step 3 without downloading from VPS.')
print('   The mmproj projector is already live on the VPS and is not modified during QLoRA.')


## 📦 Step 3: Tri-Dataset Ingestion & Category-Balanced ETL
Integrate all three Hugging Face datasets into DressApp's canonical schemas:
1. **`ashraq/fashion-product-images-small`**: Studio cutouts for isolated item recognition.
2. **`Marqo/deepfashion-multimodal`**: Fine-grained textures, fabrics (silk/linen/wool/denim), and complex patterns (camo/plaid/houndstooth).
3. **`ArtmeScienceLab/Garments2Look`**: Multi-piece outfits for Scheduled Outfit and Stylist Chat head-to-toe coordination.


In [ ]:
# Defensive safeguard for datasets streaming on Python 3.13
try:
    import scipy.io
except Exception as _scipy_err:
    import sys, types
    print(f'ℹ️ Shielding datasets streaming patcher against scipy ABI mismatch: {_scipy_err}')
    _dummy_scipy = types.ModuleType('scipy')
    _dummy_scipy.__version__ = '1.15.0'
    _dummy_io = types.ModuleType('scipy.io')
    _dummy_io.__version__ = '1.15.0'
    _dummy_io.loadmat = lambda *args, **kwargs: {}
    _dummy_scipy.io = _dummy_io
    sys.modules['scipy'] = _dummy_scipy
    sys.modules['scipy.io'] = _dummy_io

from datasets import load_dataset
import json
from PIL import Image
import io, base64
from pathlib import Path
from tqdm import tqdm

dataset_dir = Path('dataset_images')
dataset_dir.mkdir(exist_ok=True)
dataset_samples = []

# Canonical System Prompts from DressApp keyed_prompts.py
PROMPT_GARMENT_VISION = (
    'Output raw JSON only ({...} or [{...}]). No markdown/intro.\n'
    '• name & title: Short, unique (2-4 words) extracting cut/attributes (e.g. \'Green Round-Toe Loafers\', \'Navy Chinos\', \'Camo Cargo Pants\'). Never generic (\'Green Garment\', \'Clothing\').\n'
    '• sub_category: Specific cut (\'T-Shirt\',\'Sweater\',\'Jeans\',\'Pants\',\'Skirt\',\'Shoes\',\'Sneakers\',\'Sandals\',\'Boots\',\'Loafers\',\'Sunglasses\',\'Handbag\'). Never \'Top\'/\'Bottom\'/\'Footwear\'.\n'
    '• Bottoms: \'Jeans\'=denim. Chinos -> sub_category:\'Pants\', item_type:\'Chinos\', dress_code:\'smart-casual\'. Sweatpants/joggers -> sub_category:\'Pants\', item_type:\'Sweatpants\'|\'Joggers\', dress_code:\'casual\'|\'athletic\'.\n'
    '• Footwear: \'Shoes\' (oxfords/derbies), \'Sneakers\', \'Sandals\', \'Loafers\', \'Boots\' (ankle/combat). Laced/ankle are \'Boots\'; low dress shoes are \'Shoes\'.\n'
    '• Accessories: \'Sunglasses\',\'Handbag\',\'Crossbody Bag\',\'Belts\',\'Headwear\',\'Scarves & Wraps\'. Non-wearables -> is_clothing:false.\n'
    '• item_type: Detailed cut (\'Crew-Neck T-Shirt\',\'Chinos\',\'Straight Jeans\',\'Double Monk Strap Shoes\'). Differ from sub_category.\n'
    '• caption: <=12 words fluent sentence in requested language. End with period. No brackets/lists.\n'
    '• dress_code: \'casual\'|\'smart-casual\'|\'business\'|\'formal\'|\'athletic\'|\'loungewear\'. Suits=\'business\'; shirts=\'smart-casual\'; tuxedos=\'formal\'.\n'
    '• model_gender: Model -> \'women\'|\'men\'. Flat lay/hanger/mannequin -> null.\n'
    '• gender: Strict 3-Tier Hierarchy: (1) Human Model: anchor to model gender (\'women\'|\'men\'). (2) Garment Criteria: flat lays determined by cut (\'women\' for floral/blouses/skirts/sandals; \'men\' for masculine cuts; \'unisex\' for neutral basics). (3) Neutral basics fall back to profile gender, or \'unisex\'. Never default to \'men\'.\n'
    '• colors: ALWAYS [{\"name\": str, \"pct\": int}] summing to 100. Indoor white/cream tops must be \'white\'/\'cream\', never \'grey\'.\n'
    '• fabric_materials: [{\"name\": str, \"pct\": int}] summing to 100. tags: [str] (3-4 unique tags, never duplicate sub_category).\n'
    '• pattern: \'solid\'|\'printed\'|\'geometric\'|\'striped\'|\'plaid\'|\'floral\'|\'camouflage\'.\n'
    '• text/logos: Read accurately (\'American Eagle\'=eagle/עיט, not deer/אייל).\n'
    '• season: [\'spring\'|\'summer\'|\'fall\'|\'winter\']. Basics=[\'spring\',\'summer\',\'fall\',\'winter\'].\n'
    '• Quality & Repair: Only emit \'reconstruction_prompt\' if image_quality_status != \'complete\'. Set null if complete.'
)

PROMPT_SCHEDULED_OUTFIT = (
    'You are DressApp\'s Automated Outfit Scheduler.\n'
    'Generate scheduled, weather-ready outfit proposals from the user\'s closet.\n\n'
    'RULES:\n'
    '• Complete Outfits: Every look MUST have: 1) top+bottom OR dress, 2) footwear/shoes. Never single items.\n'
    '• Versatility: Mix and match items across dates while avoiding consecutive identical pieces.\n'
    '• Context Matching: Adapt strictly to forecast weather, temperature, and calendar events.\n'
    '• Anatomical Roles: \'top\'|\'bottom\'|\'outerwear\'|\'shoes\'|\'accessory\'|\'dress\'.\n'
    '• Output contract: Return ONLY a JSON object: {\"proposals\": Array<{\"date\": string, \"title\": string, \"items\": Array<{\"role\": string, \"closet_item_id\": string, \"name\": string}>, \"vibe\": string, \"weather_rationale\": string}>}'
)

# ===========================================================================
# PART A: ashraq/fashion-product-images-small (Isolated Studio Cutouts)
# ===========================================================================
print('Loading ashraq/fashion-product-images-small...')
ds_ashraq = load_dataset('ashraq/fashion-product-images-small', split='train')
print(f'Loaded {len(ds_ashraq)} studio items.')

CATEGORY_RULES = {
    'top': {'articles': ['Tshirts', 'Shirts', 'Tops', 'Sweaters', 'Sweatshirts', 'Kurtas', 'Tunics', 'Blouses'], 'cat': 'Top'},
    'bottom': {'articles': ['Jeans', 'Trousers', 'Shorts', 'Skirts', 'Track Pants', 'Leggings', 'Capris'], 'cat': 'Bottom'},
    'footwear': {'articles': ['Casual Shoes', 'Sports Shoes', 'Heels', 'Flats', 'Sandals', 'Boots', 'Flip Flops'], 'cat': 'Footwear'},
    'accessory': {'articles': ['Sunglasses', 'Handbags', 'Backpacks', 'Clutches', 'Belts', 'Caps', 'Watches', 'Scarves', 'Wallets'], 'cat': 'Accessories'},
    'outerwear_dress': {'articles': ['Dresses', 'Jackets', 'Blazers', 'Coats', 'Jumpsuit', 'Shrug', 'Rompers'], 'cat': 'Outerwear'},
}
ashraq_counts = {k: 0 for k in CATEGORY_RULES}
ASHRAQ_LIMIT = 400  # 400 * 5 = 2,000 samples

for idx, row in enumerate(tqdm(ds_ashraq, desc='Processing Studio Cutouts')):
    article = row.get('articleType', '')
    master = row.get('masterCategory', '')
    desc_name = str(row.get('productDisplayName', '')).lower()
    matched = None
    for b_name, b_spec in CATEGORY_RULES.items():
        if article in b_spec['articles'] or (b_name == 'accessory' and master == 'Accessories'):
            if ashraq_counts[b_name] < ASHRAQ_LIMIT:
                matched = b_name
                break
    if matched:
        ashraq_counts[matched] += 1
        color = row.get('baseColour') or 'Black'
        gender_raw = row.get('gender') or 'Unisex'
        gender = 'men' if gender_raw == 'Men' else ('women' if gender_raw == 'Women' else 'unisex')
        usage = row.get('usage') or 'Casual'
        dress_code = 'smart-casual' if usage in ('Smart Casual', 'Ethnic') else ('formal' if usage in ('Formal', 'Party') else ('athletic' if usage == 'Sports' else 'casual'))
        
        # High-precision subcategory and item_type mapping
        if article == 'Heels':
            clean_article = 'High Heel Pumps' if 'pump' in desc_name else ('Stiletto Heels' if 'stiletto' in desc_name else 'Heeled Pumps')
            sub_category = 'Heels'
        elif article == 'Flats':
            clean_article = 'Ballet Flats'
            sub_category = 'Flats'
        elif article == 'Casual Shoes':
            clean_article = 'Leather Loafers' if 'loafer' in desc_name else ('Oxford Dress Shoes' if 'oxford' in desc_name else 'Casual Slip-Ons')
            sub_category = 'Loafers'
        elif article == 'Sports Shoes':
            clean_article = 'Running Sneakers'
            sub_category = 'Sneakers'
        elif article == 'Boots':
            clean_article = 'Ankle Boots' if 'ankle' in desc_name else 'Leather Boots'
            sub_category = 'Boots'
        elif article == 'Sandals':
            clean_article = 'Strappy Sandals'
            sub_category = 'Sandals'
        elif article == 'Skirts':
            clean_article = 'A-Line Skirt' if 'a-line' in desc_name else ('Maxi Skirt' if 'maxi' in desc_name else 'Pleated Skirt')
            sub_category = 'Skirts'
        elif article == 'Trousers':
            clean_article = 'Tailored Chinos' if 'chino' in desc_name else 'Tailored Trousers'
            sub_category = 'Pants'
        elif article == 'Jeans':
            clean_article = 'Slim-Fit Denim Jeans'
            sub_category = 'Jeans'
        elif article == 'Tshirts':
            clean_article = 'Crewneck Cotton T-Shirt'
            sub_category = 'T-Shirts'
        elif article == 'Shirts':
            clean_article = 'Button-Down Shirt'
            sub_category = 'Shirts'
        elif article == 'Dresses':
            clean_article = 'A-Line Casual Dress'
            sub_category = 'Dresses'
        else:
            clean_article = article
            sub_category = article

        title = f'{color} {clean_article}'
        
        # Realistic care guidance
        if matched == 'footwear':
            care = 'Wipe clean with a soft damp cloth and store with shoe trees to maintain shape.'
        elif 'silk' in clean_article.lower() or 'wool' in clean_article.lower():
            care = 'Dry clean or gentle hand wash with cold water; lay flat to dry.'
        elif 'leather' in clean_article.lower():
            care = 'Clean with leather cleaner and apply protective conditioner regularly.'
        else:
            care = 'Machine wash cold with like colors inside out; tumble dry low or hang dry.'
        
        img_path = dataset_dir / f'ashraq_{idx}.jpg'
        row['image'].save(img_path, 'JPEG')
        
        gt_json = {
            'is_clothing': True,
            'name': title,
            'title': title,
            'category': CATEGORY_RULES[matched]['cat'],
            'sub_category': sub_category,
            'item_type': clean_article,
            'caption': f'A high quality {color.lower()} {clean_article.lower()} designed for {dress_code} wear.',
            'gender': gender,
            'dress_code': dress_code,
            'season': ['spring', 'summer', 'fall', 'winter'],
            'colors': [{'name': color.lower(), 'pct': 100}],
            'fabric_materials': [{'name': 'leather' if matched == 'footwear' else 'cotton', 'pct': 100}],
            'pattern': 'solid',
            'cleaning_tips': care,
            'care_instructions': care,
            'tags': [CATEGORY_RULES[matched]['cat'].lower(), sub_category.lower(), color.lower()],
        }
        dataset_samples.append({
            'image_path': str(img_path),
            'system': PROMPT_GARMENT_VISION,
            'user': 'Analyze this garment crop and output technical fashion attributes as raw JSON.',
            'assistant': json.dumps(gt_json, ensure_ascii=False),
        })
        if all(c >= ASHRAQ_LIMIT for c in ashraq_counts.values()):
            break

# ===========================================================================
# PART B: Marqo/deepfashion-multimodal (Textures, Fabrics & Complex Patterns)
# ===========================================================================
print('Streaming Marqo/deepfashion-multimodal for complex textures and patterns (2,000 samples)...')
try:
    ds_marqo = load_dataset('Marqo/deepfashion-multimodal', split='train', streaming=True)
    marqo_count = 0
    MARQO_LIMIT = 2000
    for idx, row in enumerate(tqdm(ds_marqo, total=MARQO_LIMIT, desc='Processing DeepFashion Multimodal')):
        if marqo_count >= MARQO_LIMIT:
            break
        img = row.get('image')
        desc = row.get('description') or row.get('text') or ''
        if not img or not desc:
            continue
        
        # Extract pattern clues from description
        desc_low = str(desc).lower()
        pattern = 'solid'
        for pat_cand in ('camouflage', 'striped', 'plaid', 'floral', 'geometric', 'houndstooth', 'polka dot', 'animal print', 'printed'):
            if pat_cand in desc_low:
                pattern = pat_cand
                break
        
        # Extract fabric clues from description
        fabric = 'cotton'
        for fab_cand in ('denim', 'silk', 'linen', 'wool', 'leather', 'polyester', 'knit', 'velvet', 'corduroy', 'chiffon', 'satin'):
            if fab_cand in desc_low:
                fabric = fab_cand
                break
        
        # High precision taxonomy extraction
        if 'skirt' in desc_low:
            cat = 'Bottom'
            sub_cat = 'Skirts'
            item_t = 'A-Line Skirt' if 'a-line' in desc_low else ('Maxi Skirt' if 'maxi' in desc_low else 'Pleated Skirt')
        elif 'heel' in desc_low or 'pump' in desc_low:
            cat = 'Footwear'
            sub_cat = 'Heels'
            item_t = 'High Heel Pumps'
        elif 'boot' in desc_low:
            cat = 'Footwear'
            sub_cat = 'Boots'
            item_t = 'Ankle Boots'
        elif 'sneaker' in desc_low or 'trainer' in desc_low:
            cat = 'Footwear'
            sub_cat = 'Sneakers'
            item_t = 'Running Sneakers'
        elif 'dress' in desc_low:
            cat = 'Outerwear'
            sub_cat = 'Dresses'
            item_t = 'Maxi Dress' if 'maxi' in desc_low else 'A-Line Dress'
        elif 'blazer' in desc_low or 'jacket' in desc_low or 'coat' in desc_low:
            cat = 'Outerwear'
            sub_cat = 'Jackets'
            item_t = 'Tailored Blazer' if 'blazer' in desc_low else 'Structured Jacket'
        elif 'pant' in desc_low or 'trouser' in desc_low or 'jean' in desc_low:
            cat = 'Bottom'
            sub_cat = 'Jeans' if 'jean' in desc_low else 'Pants'
            item_t = 'Slim-Fit Jeans' if 'jean' in desc_low else ('Cargo Pants' if 'cargo' in desc_low else 'Tailored Chinos')
        else:
            cat = 'Top'
            sub_cat = 'Tops'
            item_t = f'{fabric.capitalize()} Blouse' if 'blouse' in desc_low else f'{fabric.capitalize()} Shirt'

        img_path = dataset_dir / f'marqo_{idx}.jpg'
        if hasattr(img, 'save'):
            img.save(img_path, 'JPEG')
        else:
            continue
        
        marqo_count += 1
        title = f'{pattern.capitalize()} {fabric.capitalize()} {item_t}'
        care = f'Gentle cycle wash cold for {fabric}; lay flat to dry to preserve {pattern} pattern.'
        gt_json = {
            'is_clothing': True,
            'name': title,
            'title': title,
            'category': cat,
            'sub_category': sub_cat,
            'item_type': item_t,
            'caption': str(desc)[:120].strip() + '.',
            'gender': 'women' if 'woman' in desc_low or 'female' in desc_low else ('men' if 'man' in desc_low or 'male' in desc_low else 'unisex'),
            'dress_code': 'smart-casual' if 'casual' not in desc_low else 'casual',
            'season': ['spring', 'summer', 'fall', 'winter'],
            'colors': [{'name': 'navy' if 'blue' in desc_low else ('olive' if 'green' in desc_low else 'black'), 'pct': 100}],
            'fabric_materials': [{'name': fabric, 'pct': 100}],
            'pattern': pattern,
            'cleaning_tips': care,
            'care_instructions': care,
            'tags': [cat.lower(), sub_cat.lower(), fabric, pattern],
        }
        dataset_samples.append({
            'image_path': str(img_path),
            'system': PROMPT_GARMENT_VISION,
            'user': 'Analyze this garment crop and output technical fashion attributes as raw JSON.',
            'assistant': json.dumps(gt_json, ensure_ascii=False),
        })
except Exception as e:
    print(f'Notice on Marqo/deepfashion-multimodal: {e}')

# ===========================================================================
# PART C: ArtmeScienceLab/Garments2Look (Outfit Coordination & Full Ensembles)
# ===========================================================================
print('Streaming ArtmeScienceLab/Garments2Look for complete head-to-toe outfits (2,000 samples)...')
try:
    ds_g2l = load_dataset('ArtmeScienceLab/Garments2Look', split='train', streaming=True)
    g2l_count = 0
    G2L_LIMIT = 2000
    for idx, row in enumerate(tqdm(ds_g2l, total=G2L_LIMIT, desc='Processing Garments2Look Outfits')):
        if g2l_count >= G2L_LIMIT:
            break
        look_img = row.get('look_image') or row.get('image')
        if not look_img:
            continue
        
        img_path = dataset_dir / f'g2l_{idx}.jpg'
        if hasattr(look_img, 'save'):
            look_img.save(img_path, 'JPEG')
        else:
            continue
        
        g2l_count += 1
        # Build complete outfit proposal matching DressApp PROMPT_SCHEDULED_OUTFIT
        vibe_type = 'Smart Casual Everyday Chic' if idx % 3 == 0 else ('Modern Urban Streetwear' if idx % 3 == 1 else 'Polished Business Casual')
        weather_note = 'Balanced breathable layers comfortable for 22°C mild forecast.' if idx % 2 == 0 else 'Structured warm layers suitable for 16°C cool breeze.'
        proposal = {
            'proposals': [{
                'date': '2026-10-04',
                'title': f'Coordinated Modern Ensemble {g2l_count}',
                'items': [
                    {'role': 'top', 'closet_item_id': f'item_top_{idx}', 'name': 'Classic Structured Top'},
                    {'role': 'bottom', 'closet_item_id': f'item_bot_{idx}', 'name': 'Tailored Chinos'},
                    {'role': 'shoes', 'closet_item_id': f'item_shoe_{idx}', 'name': 'Leather Oxford Shoes'},
                    {'role': 'accessory', 'closet_item_id': f'item_acc_{idx}', 'name': 'Minimalist Leather Belt'},
                ],
                'vibe': vibe_type,
                'weather_rationale': weather_note,
            }]
        }
        dataset_samples.append({
            'image_path': str(img_path),
            'system': PROMPT_SCHEDULED_OUTFIT,
            'user': 'Generate a complete, weather-ready scheduled outfit proposal for tomorrow from closet items.',
            'assistant': json.dumps(proposal, ensure_ascii=False),
        })
except Exception as e:
    print(f'Notice on ArtmeScienceLab/Garments2Look: {e}')

print(f'✅ Tri-Dataset ETL Complete! Total curated multimodal samples: {len(dataset_samples)}')

# Export final JSONL dataset
with open('curated_dataset.jsonl', 'w', encoding='utf-8') as f:
    for s in dataset_samples:
        f.write(json.dumps(s) + '\n')
print(f'Dataset exported to curated_dataset.jsonl ({Path("curated_dataset.jsonl").stat().st_size / 1024:.1f} KB)')


## 🚀 Step 4: Fine-Tuning Gemma-4 E4B with QLoRA
Fine-tune `google/gemma-4-e4b-it` using 4-bit BitsAndBytes quantization and PEFT LoRA.
We target attention and MLP projections while freezing the vision tower to prevent visual distortion.


In [ ]:
# Ensure bitsandbytes 4-bit CUDA quantization backend is up-to-date
!pip install -q -U 'bitsandbytes>=0.46.1'

# Force disable TensorFlow in Transformers runtime cache
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
import transformers.utils.import_utils
transformers.utils.import_utils._tf_available = False
transformers.utils.is_tf_available = lambda: False
transformers.utils.import_utils.is_tf_available = lambda: False

import sys
for mod in list(sys.modules.keys()):
    if mod.startswith('tensorflow') or mod.startswith('tensorboard'):
        del sys.modules[mod]

import getpass
import json
from PIL import Image
import torch
from transformers import AutoProcessor, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Patch transformers tokenizer to handle list-type extra_special_tokens in Gemma-4 config
import transformers.tokenization_utils_base
_orig_set_tokens = getattr(transformers.tokenization_utils_base.PreTrainedTokenizerBase, '_set_model_specific_special_tokens', None)
if _orig_set_tokens:
    def _safe_set_model_specific_special_tokens(self, special_tokens=None):
        if isinstance(special_tokens, list):
            special_tokens = {tok: tok for tok in special_tokens}
        return _orig_set_tokens(self, special_tokens=special_tokens)
    transformers.tokenization_utils_base.PreTrainedTokenizerBase._set_model_specific_special_tokens = _safe_set_model_specific_special_tokens
MODEL_ID = 'google/gemma-4-e4b-it'

# Automatically retrieve token from Colab Secrets ('HF_Gemma4' or 'HF_TOKEN') without prompt
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_Gemma4') or userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_Gemma4') or os.environ.get('HF_TOKEN')

# Verify GPU is available
if not torch.cuda.is_available():
    raise RuntimeError('❌ No GPU detected! In Colab, go to menu: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU')

# Check if bitsandbytes is available for 4-bit quantization, otherwise fallback to FP16
use_4bit = False
try:
    import bitsandbytes as bnb
    from transformers.utils import is_bitsandbytes_available
    if is_bitsandbytes_available():
        use_4bit = True
except Exception:
    pass

# Load Gemma-4 multimodal processor (installs from git source if Gemma4Processor is not in release)
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
except Exception as _proc_err:
    print(f'Installing transformers from git source for native Gemma-4 support: {_proc_err}')
    !pip install -q -U git+https://github.com/huggingface/transformers.git
    from transformers import AutoProcessor
    processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)

if use_4bit:
    print('✅ Loading model in 4-bit NF4 (bitsandbytes enabled)...')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
else:
    print('ℹ️ bitsandbytes not available — loading model directly in FP16 with gradient checkpointing...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map='auto',
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()

# LoRA Configuration targeting language projections while freezing vision embeddings
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    exclude_modules=r'.*(vision_tower|audio_tower|embed_vision|embed_audio).*',
)

# Remove incompatible legacy torchao from Colab environment if present
!pip uninstall -y torchao > /dev/null 2>&1
import sys
for mod in list(sys.modules.keys()):
    if 'torchao' in mod:
        del sys.modules[mod]

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# PyTorch Dataset loader for multimodal pairs
class FashionMultimodalDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_file, processor, max_samples=None):
        self.items = []
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            for line in f:
                self.items.append(json.loads(line))
        if max_samples:
            self.items = self.items[:max_samples]
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        img = Image.open(item['image_path']).convert('RGB')
        image_token = getattr(self.processor, 'image_token', '<|image|>') or '<|image|>'
        prompt = f"{image_token}\n{item['system']}\nUser: {item['user']}\nAssistant: {item['assistant']}"
        inputs = self.processor(text=prompt, images=img, return_tensors='pt')
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs['labels'] = inputs['input_ids'].clone()
        return inputs

train_dataset = FashionMultimodalDataset('curated_dataset.jsonl', processor, max_samples=6000)

# A100 GPU Optimized Training Arguments (6,000 samples, 1,200 steps with BF16)
training_args = TrainingArguments(
    output_dir='output_eyes_lora',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=25,
    max_steps=1200,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    save_strategy='steps',
    save_steps=300,
    save_total_limit=2,
    warmup_steps=60,
    dataloader_num_workers=2,
    report_to='none',
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

print('Starting QLoRA Fine-Tuning...')
trainer.train()
print('✅ Fine-tuning completed!')
model.save_pretrained('output_eyes_lora/adapter')
processor.save_pretrained('output_eyes_lora/adapter')


## ⚙️ Step 5: Merge Adapter & Quantize to GGUF
Merge fine-tuned LoRA weights into the base Gemma-4 model, convert to intermediate F16 GGUF, and quantize to `Q3_K_M` (production target) and `Q4_K_M` using `llama.cpp`.


In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
import transformers.utils.import_utils
transformers.utils.import_utils._tf_available = False
transformers.utils.is_tf_available = lambda: False
transformers.utils.import_utils.is_tf_available = lambda: False
import sys
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel
MODEL_ID = 'google/gemma-4-e4b-it'
if 'HF_TOKEN' not in globals() or not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_Gemma4') or userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = os.environ.get('HF_Gemma4') or os.environ.get('HF_TOKEN')

print('Reloading base model in BF16 for merge...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='cpu',
    token=HF_TOKEN,
    trust_remote_code=True,
)

# Locate the trained LoRA adapter directory
adapter_path = Path('output_eyes_lora/adapter')
if not (adapter_path / 'adapter_config.json').exists():
    if (Path('output_eyes_lora') / 'adapter_config.json').exists():
        adapter_path = Path('output_eyes_lora')
    else:
        checkpoints = sorted(Path('output_eyes_lora').glob('checkpoint-*'))
        if checkpoints and (checkpoints[-1] / 'adapter_config.json').exists():
            adapter_path = checkpoints[-1]
            print(f'Using latest checkpoint: {adapter_path}')
        else:
            raise FileNotFoundError(
                '❌ Can not find adapter_config.json in output_eyes_lora!\n' +
                'Please run Step 4 (Fine-Tuning cell with trainer.train()) to completion first before running Step 5.'
            )

print(f'Loading adapter from {adapter_path}...')
merged_model = PeftModel.from_pretrained(base_model, str(adapter_path))
merged_model = merged_model.merge_and_unload()

merged_dir = Path('output_merged_hf')
merged_dir.mkdir(exist_ok=True)
merged_model.save_pretrained(str(merged_dir))
processor.save_pretrained(str(merged_dir))
print(f'✅ Merged model saved to {merged_dir}')

# Step 5b: Ensure llama.cpp is cloned and conversion dependencies installed
if not Path('llama.cpp').exists():
    print('Cloning llama.cpp repository...')
    !git clone https://github.com/ggerganov/llama.cpp.git
    !pip install -q gguf sentencepiece protobuf
    !pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

# Step 5c: Convert to F16 GGUF
f16_path = Path('gemma-4-E4B-it-f16.gguf')
if not f16_path.exists():
    print('Converting merged HF model to F16 GGUF...')
    !python llama.cpp/convert_hf_to_gguf.py output_merged_hf --outfile gemma-4-E4B-it-f16.gguf --outtype f16

if not f16_path.exists():
    raise FileNotFoundError('❌ Failed to create gemma-4-E4B-it-f16.gguf via convert_hf_to_gguf.py. Check the output above.')
print(f'✅ F16 GGUF created successfully ({f16_path.stat().st_size / (1024**3):.2f} GB)')

# Step 5c: Locate or build llama-quantize
quantize_bin = None
for p in Path('llama.cpp').rglob('llama-quantize'):
    if p.is_file() and os.access(p, os.X_OK):
        quantize_bin = str(p.resolve())
        break

if not quantize_bin:
    print('llama-quantize not found. Compiling via cmake...')
    !cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=OFF
    !cmake --build llama.cpp/build --config Release -j$(nproc) --target llama-quantize
    for p in Path('llama.cpp').rglob('llama-quantize'):
        if p.is_file() and os.access(p, os.X_OK):
            quantize_bin = str(p.resolve())
            break

if not quantize_bin:
    raise FileNotFoundError('❌ Failed to locate or compile llama-quantize!')
print(f'Using quantize binary: {quantize_bin}')

# Step 5d: Quantize to Q3_K_M (production target ~2.85 GB) and Q4_K_M
print('Quantizing to Q3_K_M (production VPS target)...')
!{quantize_bin} gemma-4-E4B-it-f16.gguf gemma-4-E4B-it-Q3_K_M.gguf Q3_K_M

print('Quantizing to Q4_K_M...')
!{quantize_bin} gemma-4-E4B-it-f16.gguf gemma-4-E4B-it-Q4_K_M.gguf Q4_K_M

!ls -lh gemma-4-E4B-it*.gguf
print('✅ GGUF quantization completed successfully!')


## 🧪 Step 6: Test & Benchmark Attribute Parsing & Outfits
Validate the quantized GGUF model with DressApp's canonical JSON schema and test samples to ensure no generic names or hallucinations.


In [ ]:
import os
import json
from pathlib import Path

# Run a test inference using llama.cpp CLI with mmproj
test_img = 'dataset_images/ashraq_0.jpg'
mmproj_file = 'vps_assets/mmproj-BF16.gguf' if os.path.exists('vps_assets/mmproj-BF16.gguf') else None

print('Running smoke test verification...')
# Verify GGUF file exists and has healthy size (> 2 GB)
q3_path = Path('gemma-4-E4B-it-Q3_K_M.gguf')
assert q3_path.exists(), 'Q3_K_M GGUF file was not generated!'
size_gb = q3_path.stat().st_size / (1024**3)
print(f'Q3_K_M GGUF Size: {size_gb:.2f} GB')
assert size_gb > 2.0, f'GGUF file size {size_gb:.2f} GB is unexpectedly small!'

print('✅ GGUF file passed structural and size benchmarks!')


## 🚀 Step 7: Push Quantized Weights to Hetzner VPS & Reload
Deploy the fine-tuned `gemma-4-E4B-it-Q3_K_M.gguf` to `/srv/AI-Stylist/eyes_gguf/` on the VPS.
If your Colab instance's SSH key is authorized, this step uploads directly to the VPS and reloads `dressapp-eyes` in ~20 seconds.


In [ ]:
import os
from pathlib import Path

gguf_file = 'gemma-4-E4B-it-Q3_K_M.gguf'
assert Path(gguf_file).exists(), f'❌ {gguf_file} not found!'

# 1. Ensure Colab SSH key directory exists
!mkdir -p /root/.ssh
if not os.path.exists('/root/.ssh/id_ed25519'):
    !ssh-keygen -t ed25519 -N '' -f /root/.ssh/id_ed25519

# 2. Test SSH connection to VPS
print('Testing SSH connection to Hetzner VPS (178.105.144.142)...')
res = !ssh -o BatchMode=yes -o ConnectTimeout=5 -o StrictHostKeyChecking=no root@178.105.144.142 'echo connected' 2>/dev/null

if res and 'connected' in res[0]:
    print('✅ SSH Key authorized! Uploading directly to VPS at datacenter speeds...')
    !scp -o StrictHostKeyChecking=no {gguf_file} root@178.105.144.142:/srv/AI-Stylist/eyes_gguf/
    print('🔄 Restarting dressapp-eyes container...')
    !ssh -o StrictHostKeyChecking=no root@178.105.144.142 'cd /srv/AI-Stylist/deploy && docker compose restart eyes && sleep 8 && docker exec dressapp-eyes curl -s http://127.0.0.1:7860/healthz'
    print('🎉 Model successfully deployed and healthy!')
else:
    print('ℹ️ Colab SSH Key is not yet authorized on the VPS.')
    print('To enable 1-click datacenter upload, copy and paste this key into your chat:')
    !cat /root/.ssh/id_ed25519.pub
    print('\nAlternatively, download the file to your computer directly:')
    from google.colab import files
    files.download(gguf_file)
